In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content
!rm -rf yolov5
!git clone https://github.com/ultralytics/yolov5.git
%cd /content/yolov5
!pip install -qr requirements.txt
!pip install -q roboflow

/content
Cloning into 'yolov5'...
remote: Enumerating objects: 17847, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 17847 (delta 25), reused 6 (delta 6), pack-reused 17806 (from 3)
Receiving objects: 100% (17847/17847), 16.99 MiB | 16.65 MiB/s, done.
Resolving deltas: 100% (12166/12166), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 96.4 MB/s eta 0:00:00


In [3]:
%cd /content

import os
import zipfile
import glob
import shutil

ZIP_PATH = "/content/KitchenTools_Detection.v4i.yolov5pytorch.zip"
EXTRACT_DIR = "/content/dataset"

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("압축 해제 완료")
print("폴더 내부:", os.listdir(EXTRACT_DIR))

/content


BadZipFile: File is not a zip file

In [4]:
import zipfile
import os
import shutil

ZIP_PATH = "/content/KitchenTools_Detection.v4i.yolov5pytorch.zip"
EXTRACT_DIR = "/content/dataset"

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("압축 해제 완료")
print(os.listdir(EXTRACT_DIR))

압축 해제 완료
['valid', 'README.dataset.txt', 'data.yaml', 'test', 'README.roboflow.txt', 'train']


In [5]:
import yaml
from glob import glob
import os

yaml_candidates = glob('/content/dataset/**/data.yaml', recursive=True)
print("찾은 YAML:", yaml_candidates)

DATA_YAML = yaml_candidates[0]

with open(DATA_YAML, 'r') as f:
    data = yaml.safe_load(f)

base_dir = os.path.dirname(DATA_YAML)

def fix_path(p):
    if os.path.isabs(p):
        return p
    return os.path.join(base_dir, p)

data['train'] = fix_path(data['train'])
data['val'] = fix_path(data['val'])

if 'test' in data:
    data['test'] = fix_path(data['test'])

NEW_YAML = "/content/dataset/data_fixed.yaml"

with open(NEW_YAML, 'w') as f:
    yaml.dump(data, f)

print("수정된 YAML:", NEW_YAML)
print(data)

찾은 YAML: ['/content/dataset/data.yaml']
수정된 YAML: /content/dataset/data_fixed.yaml
{'train': '/content/dataset/../train/images', 'val': '/content/dataset/../valid/images', 'test': '/content/dataset/../test/images', 'nc': 5, 'names': ['blade', 'fork', 'handle', 'ladle', 'plate'], 'roboflow': {'workspace': 's-workspace-wham6', 'project': 'kitchentools_detection-go49d-eyc12', 'version': 4, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/s-workspace-wham6/kitchentools_detection-go49d-eyc12/dataset/4'}}


In [6]:
import yaml

with open("/content/dataset/data_fixed.yaml",'r') as f:
    data = yaml.safe_load(f)

print("train:",data['train'])
print("val:",data['val'])
print("classes:",data['names'])
print("nc:",data['nc'])

train: /content/dataset/../train/images
val: /content/dataset/../valid/images
classes: ['blade', 'fork', 'handle', 'ladle', 'plate']
nc: 5


In [7]:
import os
from datetime import datetime

RUN_NAME = datetime.now().strftime("%Y%m%d_%H%M%S")

SAVE_DIR = f"/content/drive/MyDrive/yolo_training_{RUN_NAME}"

os.makedirs(SAVE_DIR, exist_ok=True)

print("자동 저장 위치:", SAVE_DIR)

자동 저장 위치: /content/drive/MyDrive/yolo_training_20260316_112847


In [8]:
%cd /content/yolov5

!python train.py \
--img 640 \
--batch 16 \
--epochs 100 \
--data /content/dataset/data_fixed.yaml \
--weights yolov5s.pt \
--name kitchen_tools

/content/yolov5
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2026-03-16 11:29:12.003207: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773660552.026141    2919 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773660552.033131    2919 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for 

In [9]:
import os
import yaml
from glob import glob

# data.yaml 찾기
yaml_candidates = glob('/content/dataset/**/data.yaml', recursive=True)
print("찾은 YAML 후보:", yaml_candidates)

assert len(yaml_candidates) > 0, "data.yaml을 찾지 못함"

DATA_YAML = yaml_candidates[0]
print("사용할 YAML:", DATA_YAML)

with open(DATA_YAML, 'r') as f:
    data = yaml.safe_load(f)

base_dir = os.path.dirname(DATA_YAML)
print("base_dir:", base_dir)

def smart_fix_path(p):
    if p is None:
        return None

    # 이미 절대경로면 그대로
    if os.path.isabs(p):
        return os.path.normpath(p)

    # 1차: data.yaml 위치 기준
    candidate1 = os.path.normpath(os.path.join(base_dir, p))
    if os.path.exists(candidate1):
        return candidate1

    # 2차: /content/dataset 기준으로 다시 시도
    candidate2 = os.path.normpath(os.path.join('/content/dataset', p))
    if os.path.exists(candidate2):
        return candidate2

    # 3차: train/valid/test 같은 폴더명만 뽑아서 dataset 바로 아래로 강제 연결
    parts = p.replace("\\", "/").split("/")
    if len(parts) >= 2:
        candidate3 = os.path.normpath(os.path.join('/content/dataset', parts[-2], parts[-1]))
        parent3 = os.path.normpath(os.path.join('/content/dataset', parts[0], parts[1])) if len(parts) >= 2 else None
        if os.path.exists(candidate3):
            return candidate3
        if parent3 and os.path.exists(parent3):
            return parent3

    # 마지막 fallback
    return candidate1

data['train'] = smart_fix_path(data['train'])
data['val'] = smart_fix_path(data['val'])

if 'test' in data:
    data['test'] = smart_fix_path(data['test'])

NEW_YAML = "/content/dataset/data_fixed.yaml"
with open(NEW_YAML, 'w') as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

print("\n수정 완료")
print(data)
print("\n저장 위치:", NEW_YAML)

찾은 YAML 후보: ['/content/dataset/data.yaml']
사용할 YAML: /content/dataset/data.yaml
base_dir: /content/dataset

수정 완료
{'train': '/content/dataset/train/images', 'val': '/content/dataset/valid/images', 'test': '/content/dataset/test/images', 'nc': 5, 'names': ['blade', 'fork', 'handle', 'ladle', 'plate'], 'roboflow': {'workspace': 's-workspace-wham6', 'project': 'kitchentools_detection-go49d-eyc12', 'version': 4, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/s-workspace-wham6/kitchentools_detection-go49d-eyc12/dataset/4'}}

저장 위치: /content/dataset/data_fixed.yaml


In [10]:
import os
import yaml

with open("/content/dataset/data_fixed.yaml", "r") as f:
    data = yaml.safe_load(f)

print("train:", data['train'], "->", os.path.exists(data['train']))
print("val  :", data['val'], "->", os.path.exists(data['val']))

if 'test' in data:
    print("test :", data['test'], "->", os.path.exists(data['test']))

print("nc:", data.get('nc'))
print("names:", data.get('names'))

train: /content/dataset/train/images -> True
val  : /content/dataset/valid/images -> True
test : /content/dataset/test/images -> True
nc: 5
names: ['blade', 'fork', 'handle', 'ladle', 'plate']


In [11]:
!find /content/dataset -maxdepth 3 -type d | sort

/content/dataset
/content/dataset/test
/content/dataset/test/images
/content/dataset/test/labels
/content/dataset/train
/content/dataset/train/images
/content/dataset/train/labels
/content/dataset/valid
/content/dataset/valid/images
/content/dataset/valid/labels


/content/yolov5
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2026-03-16 11:32:48.647262: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773660768.667922    3904 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773660768.674812    3904 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773660768.692203    3904 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773660768.692234    3904 computation_placer.cc:177] computation placer a

In [ ]:
%cd /content/yolov5

!python train.py \
--img 640 \
--batch 16 \
--epochs 200 \
--data /content/dataset/data_fixed.yaml \
--weights yolov5s.pt \
--name kitchen_tools

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  with torch.cuda.amp.autocast(amp):
    109/199      4.33G    0.01948    0.01158   0.002022         48        640:  79% 411/519 [02:33<00:35,  3.02it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
    109/199      4.33G    0.01948    0.01157   0.002018         50        640:  79% 412/519 [02:33<00:36,  2.97it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
    109/199      4.33G    0.01948    0.01158   0.002016         62        640:  80% 413/519 [02:33<00:32,  3.27it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
    109/1

In [1]:
!find /content/drive/MyDrive -name "best.pt" -o -name "last.pt"

find: ‘/content/drive/MyDrive’: No such file or directory


Mounted at /content/drive
